# Parametric Health Insurance Pricing Case Study

This notebook develops a **parametric health insurance pricing model** inspired by the SOA Student Research Case Study framework.

The product is designed for governments, healthcare organizations, or institutional buyers that need rapid financial support when public health indicators deteriorate beyond a predefined trigger.

## Project Objective

The objective is to design and price a parametric insurance product using:

- Health-related trigger indicators
- A PCA-based composite Health Index
- A transparent trigger threshold
- Parametric payout calculation
- Monte Carlo simulation
- Equivalence-principle premium calculation
- Sensitivity testing for pricing assumptions

## 1. Import Required Libraries

The analysis uses:

- `pandas` for data handling
- `numpy` for numerical calculation and simulation
- `matplotlib` for visualization
- `scikit-learn` for standardization and PCA

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

print("Libraries loaded successfully.")

## 2. Load the Cleaned SOA Case Study Data

The cleaned workbook should be placed in the same folder as this notebook.

Expected file name:

`soa_parametric_health_cleaned_data.xlsx`

The key input sheet is `Model_Input`, which contains health-related indicators by country and year.

In [ ]:
file_path = "soa_parametric_health_cleaned_data.xlsx"

print("Current working directory:")
print(os.getcwd())

print("\nFiles in current folder:")
print(os.listdir())

xls = pd.ExcelFile(file_path)
xls.sheet_names

In [ ]:
model_data = pd.read_excel(
    file_path,
    sheet_name="Model_Input"
)

model_data.head()

## 3. Initial Data Quality Check

Before modeling, check the structure of the dataset and confirm whether missing values exist.

This is important because PCA and Monte Carlo simulation require consistent numerical inputs.

In [ ]:
print("Dataset shape:", model_data.shape)

missing_values = model_data.isna().sum()
missing_values

## 4. Select Health Trigger Variables

The product uses three health-related indicators as trigger variables:

1. **Blood Pressure** represented by `BP Component`
2. **Diabetes Prevalence** represented by `Avg Diabetes`
3. **Cholesterol Risk** represented by `Cholesterol Ratio`

These indicators were selected because they are measurable, health-related, and suitable for constructing a transparent parametric trigger.

In [ ]:
features = ["BP Component", "Avg Diabetes", "Cholesterol Ratio"]

analysis_data = model_data[["Year", "Country"] + features].copy()

analysis_data.head()

## 5. Correlation Analysis

The correlation matrix is used to understand whether the selected health indicators move together.

A strong positive correlation suggests that the variables share common health-risk information, which supports the use of PCA to construct a composite Health Index.

In [ ]:
corr_matrix = analysis_data[features].corr()
corr_matrix

In [ ]:
plt.figure(figsize=(6, 4))
plt.imshow(corr_matrix, cmap="Blues")
plt.colorbar()

plt.xticks(range(len(features)), features, rotation=45, ha="right")
plt.yticks(range(len(features)), features)

plt.title("Correlation Matrix of Health Indicators")
plt.tight_layout()
plt.show()

## 6. Principal Component Analysis (PCA)

PCA is used to combine the three health indicators into a single composite index.

Before PCA, the variables are standardized because they are measured on different scales.

In [ ]:
X = analysis_data[features].dropna()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=1)
pca_score = pca.fit_transform(X_scaled)

explained_variance = pca.explained_variance_ratio_[0]

print(f"Explained variance of PC1: {explained_variance:.4f}")

In [ ]:
pca_loadings = pd.DataFrame(
    pca.components_.T,
    index=features,
    columns=["PC1_Loading"]
)

pca_loadings

## 7. Construct the PCA-Based Health Index

The raw PCA score is useful statistically, but it is difficult to interpret in product design.

Therefore, the PCA score is rescaled into a **0–100 Health Index** using min-max normalization:

- 0 represents the lowest observed health-risk level in the dataset.
- 100 represents the highest observed health-risk level in the dataset.

This makes the trigger and payout structure easier to explain to non-technical stakeholders.

In [ ]:
analysis_data = analysis_data.loc[X.index].copy()

analysis_data["PCA_Score"] = pca_score[:, 0]

min_score = analysis_data["PCA_Score"].min()
max_score = analysis_data["PCA_Score"].max()

analysis_data["PCA_Health_Index"] = (
    (analysis_data["PCA_Score"] - min_score) /
    (max_score - min_score)
) * 100

analysis_data.head()

In [ ]:
plt.figure(figsize=(10, 5))

for country in analysis_data["Country"].unique():
    temp = analysis_data[analysis_data["Country"] == country]
    plt.plot(
        temp["Year"],
        temp["PCA_Health_Index"],
        marker="o",
        label=country
    )

plt.title("PCA-Based Health Index Over Time")
plt.xlabel("Year")
plt.ylabel("Health Index (0-100)")
plt.legend()
plt.tight_layout()
plt.show()

## 8. Define Trigger Event and Payout Formula

The trigger threshold is set at the **90th percentile** of the historical Health Index distribution.

This means the payout is triggered only when the Health Index reaches an unusually high-risk level.

The payout formula is:

`Payout = Payout Factor × Excess over Threshold`

where:

`Excess = max(Health Index - Trigger Threshold, 0)`

In [ ]:
threshold_percentile = 0.90
threshold = analysis_data["PCA_Health_Index"].quantile(threshold_percentile)

analysis_data["Trigger"] = np.where(
    analysis_data["PCA_Health_Index"] > threshold,
    1,
    0
)

analysis_data["Excess"] = np.maximum(
    analysis_data["PCA_Health_Index"] - threshold,
    0
)

payout_factor = 1120
analysis_data["Payout"] = payout_factor * analysis_data["Excess"]

print(f"Trigger threshold ({threshold_percentile:.0%} percentile): {threshold:.4f}")

analysis_data[[
    "Year",
    "Country",
    "PCA_Health_Index",
    "Trigger",
    "Excess",
    "Payout"
]].head()

In [ ]:
triggered_observations = analysis_data[analysis_data["Payout"] > 0][[
    "Year",
    "Country",
    "PCA_Health_Index",
    "Trigger",
    "Excess",
    "Payout"
]]

triggered_observations

## 9. Monte Carlo Simulation

Monte Carlo simulation is used to estimate:

1. The probability that the trigger event occurs.
2. The expected payout under simulated future Health Index scenarios.

The Health Index is simulated using a normal distribution based on the historical mean and standard deviation.

In [ ]:
np.random.seed(42)

n_sim = 100000

mean_index = analysis_data["PCA_Health_Index"].mean()
std_index = analysis_data["PCA_Health_Index"].std()

simulated_index = np.random.normal(
    loc=mean_index,
    scale=std_index,
    size=n_sim
)

simulated_excess = np.maximum(simulated_index - threshold, 0)
simulated_payout = payout_factor * simulated_excess

expected_payout = simulated_payout.mean()
trigger_probability = np.mean(simulated_index > threshold)

simulation_summary = pd.DataFrame({
    "Metric": [
        "Number of simulations",
        "Mean Health Index",
        "Health Index standard deviation",
        "Trigger threshold",
        "Trigger probability",
        "Expected payout"
    ],
    "Value": [
        n_sim,
        mean_index,
        std_index,
        threshold,
        trigger_probability,
        expected_payout
    ]
})

simulation_summary

## 10. Premium Calculation

The basic premium is calculated using the **equivalence principle**:

`Basic Premium = Expected Payout`

A 30% loading is added to allow for expenses, risk margin, and profit.

`Gross Premium = Basic Premium × (1 + Loading)`

In [ ]:
basic_premium = expected_payout

loading = 0.30
gross_premium = basic_premium * (1 + loading)

premium_summary = pd.DataFrame({
    "Metric": [
        "Basic Premium",
        "Loading",
        "Gross Premium"
    ],
    "Value": [
        basic_premium,
        loading,
        gross_premium
    ]
})

premium_summary

## 11. Revenue Analysis

Revenue analysis is performed for an illustrative portfolio of 100,000 covered units.

This step connects the pricing result to business sustainability.

In [ ]:
number_of_policies = 100000

expected_revenue = gross_premium * number_of_policies
expected_claim_cost = expected_payout * number_of_policies
expected_margin = expected_revenue - expected_claim_cost

revenue_summary = pd.DataFrame({
    "Metric": [
        "Number of Covered Units",
        "Expected Revenue",
        "Expected Claim Cost",
        "Expected Margin"
    ],
    "Value": [
        number_of_policies,
        expected_revenue,
        expected_claim_cost,
        expected_margin
    ]
})

revenue_summary

## 12. Sensitivity Testing

Sensitivity testing is used to assess how pricing results change under different assumptions.

The tested assumptions are:

- Trigger percentile: 85%, 90%, 95%
- Payout factor: 800, 1120, 1500
- Loading: 20%, 30%, 40%

This helps evaluate premium adequacy and product sustainability under different risk and pricing scenarios.

In [ ]:
sensitivity_results = []

threshold_percentiles = [0.85, 0.90, 0.95]
loadings = [0.20, 0.30, 0.40]
payout_factors = [800, 1120, 1500]

for p in threshold_percentiles:
    temp_threshold = analysis_data["PCA_Health_Index"].quantile(p)

    for pf in payout_factors:
        temp_excess = np.maximum(simulated_index - temp_threshold, 0)
        temp_payout = pf * temp_excess
        temp_expected_payout = temp_payout.mean()
        temp_trigger_probability = np.mean(simulated_index > temp_threshold)

        for l in loadings:
            temp_gross_premium = temp_expected_payout * (1 + l)

            sensitivity_results.append({
                "Threshold Percentile": p,
                "Payout Factor": pf,
                "Loading": l,
                "Trigger Probability": temp_trigger_probability,
                "Expected Payout": temp_expected_payout,
                "Gross Premium": temp_gross_premium
            })

sensitivity_df = pd.DataFrame(sensitivity_results)

sensitivity_df

In [ ]:
base_sensitivity = sensitivity_df[
    (sensitivity_df["Threshold Percentile"] == 0.90) &
    (sensitivity_df["Loading"] == 0.30)
]

plt.figure(figsize=(8, 5))
plt.plot(
    base_sensitivity["Payout Factor"],
    base_sensitivity["Gross Premium"],
    marker="o"
)

plt.title("Gross Premium Sensitivity to Payout Factor")
plt.xlabel("Payout Factor")
plt.ylabel("Gross Premium")
plt.tight_layout()
plt.show()

## 13. Save Results

The final outputs are saved into an Excel workbook for review and documentation.

In [ ]:
output_file = "soa_parametric_health_results.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    analysis_data.to_excel(writer, sheet_name="Model_Data", index=False)
    corr_matrix.to_excel(writer, sheet_name="Correlation_Matrix")
    pca_loadings.to_excel(writer, sheet_name="PCA_Loadings")
    simulation_summary.to_excel(writer, sheet_name="Monte_Carlo_Summary", index=False)
    premium_summary.to_excel(writer, sheet_name="Premium_Summary", index=False)
    revenue_summary.to_excel(writer, sheet_name="Revenue_Summary", index=False)
    sensitivity_df.to_excel(writer, sheet_name="Sensitivity_Testing", index=False)

print(f"Results saved successfully to: {output_file}")

## 14. Key Interpretation for Interview Discussion

The project can be explained using the following storyline:

1. **Trigger Selection**  
   Blood pressure, diabetes prevalence, and cholesterol were selected because they are measurable health indicators and are linked to healthcare stress.

2. **Correlation Analysis**  
   The selected variables showed strong relationships, which supported the use of PCA.

3. **PCA Health Index**  
   PCA was used to combine the selected variables into a single composite Health Index. The index was rescaled to 0–100 to make it easier to interpret.

4. **Trigger Design**  
   The trigger threshold was set at the 90th percentile of the historical Health Index distribution.

5. **Payout Structure**  
   The payout was defined as a function of the excess over the trigger threshold.

6. **Monte Carlo Simulation**  
   Simulation was used to estimate trigger probability and expected payout.

7. **Premium Calculation**  
   The basic premium was calculated using the equivalence principle, and a loading was added for expenses, risk margin, and profit.

8. **Sensitivity Testing**  
   Different thresholds, payout factors, and loading assumptions were tested to evaluate premium adequacy and financial sustainability.

## Model Limitations

- The payout factor is treated as a product design parameter.
- The simulation assumes the Health Index follows a normal distribution.
- The model uses public case-study indicators rather than insurer-specific claims data.
- In a real product, the payout factor and trigger threshold should be calibrated using emergency funding needs, affordability, and the insurer's risk appetite.